![alt text](images/uspas.png)
# Fundamentals of Accelerator Physics and Technology
### (with Simulations and Measurements Lab)
# Computer Lab: Dispersion and Chromaticity in Simulated Beam Transport
##### Authors: K. Ruisard, N. Evans, N. Neveu and L. Dovlatyan

## We will be simulating beam transport in simple beamlines directly in Python with Xsuite. Questions to be turned in for credit are in **bold** and numbered.
### Python Notes:
- The code cells below build the lattices, compute Twiss functions, dispersion, chromaticity, and tune footprints directly in the notebook.
- Press shift+enter to execute a cell, or use the play button at the top of the window.
- Make sure you execute cells in order, or re-execute cells if you change something at the top of the notebook.
- You can also execute the whole notebook by using 'Run all cells' under the 'Run' tab.

</br>
Also helpful: Shift+right click brings up OS/browser right-click menu, can copy image or save.

----------


## 1. Dispersion and Chromaticity

### A) Dispersion in a FODO Lattice

In other labs, we simulate beamlines without dipoles. Dipoles create dispersion ($\eta$ or $D$, depending who you talk to), which describes the evolution of transverse size for particles/beams that are off-momentum. Off-momentum particles experience a different force in the dipole than the design orbit, as they have different magnetic rigidity $B\rho=\frac{p}{q}$. Dispersion is a function of $s$ and can have a periodic matched solution, similar to the $\beta$ and $\alpha$ functions.

The initial conditions for this simulation:

| Parameter | Value |
|---|---|
| Species | Electron |
| Energy | $1\ \mathrm{GeV}$ |
| X emittance | $\epsilon_x = 6\ \mathrm{mm\,mrad}$ |
| Y emittance | $\epsilon_y = 6\ \mathrm{mm\,mrad}$ |
| Quadrupole geometric strength | $K = 0.6\ \mathrm{m}^{-2}$ |
| FODO cell length | $L = 5\ \mathrm{m}$ |

The first Xsuite lattice is a FODO cell similar to the quadrupole-focusing lab, but with a 20-degree rectangular bend inserted into one drift. The code computes the periodic optics and dispersion for the modified cell.


In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import xtrack as xt

pio.renderers.default = "notebook"

GEOMETRIC_EMITTANCE = 6e-6  # 6 mm mrad = 6e-6 m rad
SIGMA_DP_DEFAULT = 1e-3


def electron_ref(p0c=1e9):
    return xt.Particles(p0c=p0c, mass0=xt.ELECTRON_MASS_EV, q0=-1)


def build_line(elements, names):
    line = xt.Line(elements=elements, element_names=names)
    line.particle_ref = electron_ref()
    line.build_tracker()
    return line


def make_fodo_with_bend(k1=0.6, bend_angle_deg=20.0):
    bend_length = 0.5
    angle = np.deg2rad(bend_angle_deg)
    h = angle / bend_length
    return build_line(
        elements=[
            xt.Drift(length=1.0),
            xt.Quadrupole(length=0.5, k1=k1),
            xt.Drift(length=1.5),
            xt.RBend(length=bend_length, angle=angle, k0=h, h=h),
            xt.Quadrupole(length=0.5, k1=-k1),
            xt.Drift(length=1.0),
        ],
        names=["d1", "qf", "d2a", "dipo", "qd", "d3"],
    )


def twiss_table(tw):
    cols = ["name", "s", "betx", "bety", "dx", "dpx", "mux", "muy"]
    return tw.to_pandas()[cols]


def add_beam_size(tw, sigma_dp=0.0, emit_x=GEOMETRIC_EMITTANCE, emit_y=GEOMETRIC_EMITTANCE):
    df = tw.to_pandas()[["name", "s", "betx", "bety", "dx", "dpx"]].copy()
    df["sigma_x_mm"] = 1e3 * np.sqrt(df["betx"] * emit_x + (df["dx"] * sigma_dp) ** 2)
    df["sigma_y_mm"] = 1e3 * np.sqrt(df["bety"] * emit_y)
    return df


def _hover_trace(x, y, text, name, mode="lines+markers"):
    return go.Scatter(
        x=x,
        y=y,
        mode=mode,
        name=name,
        text=text,
        hovertemplate="%{text}<br>s = %{x:.4g} m<br>value = %{y:.5g}<extra></extra>",
    )


def plot_optics(tw, title="Optics"):
    names = np.asarray(tw.name, dtype=str)
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Beta functions", "Dispersion"),
    )
    fig.add_trace(_hover_trace(tw.s, tw.betx, names, "βx"), row=1, col=1)
    fig.add_trace(_hover_trace(tw.s, tw.bety, names, "βy"), row=1, col=1)
    fig.add_trace(_hover_trace(tw.s, tw.dx, names, "ηx"), row=2, col=1)
    fig.add_trace(_hover_trace(tw.s, tw.dpx, names, "ηx'"), row=2, col=1)
    fig.update_xaxes(title_text="s [m]", row=2, col=1)
    fig.update_yaxes(title_text="β [m]", row=1, col=1)
    fig.update_yaxes(title_text="dispersion [m]", row=2, col=1)
    fig.update_layout(
        title=title,
        hovermode="closest",
        template="plotly_white",
        width=900,
        height=620,
    )
    fig.show()
    return fig

print(f"xtrack version: {xt.__version__}")
print(f"plotly renderer: {pio.renderers.default}")


In [ ]:
fodo_bend = make_fodo_with_bend()
tw_fodo_bend = fodo_bend.twiss(method="4d", compute_chromatic_properties=True)

plot_optics(tw_fodo_bend, "Periodic optics for a FODO cell with one 20-degree bend");
twiss_table(tw_fodo_bend)


#### A comment on dipole edge focusing

A rectangular bend changes the transverse focusing even for an on-momentum beam. In this Xsuite model, the horizontal and vertical beta functions differ from the no-bend FODO case because the bend curvature and its rectangular geometry introduce additional focusing terms. Dispersion and chromaticity are separate effects: they matter when particles have momentum offsets.

#### Effect of dispersion on beam size

The horizontal beam size including momentum spread is

$\sigma_x^2 = \epsilon_x \beta_x + \eta_x^2 \left(\frac{\Delta p}{p_0}\right)^2$.

The vertical beam size here has no vertical-dispersion term, so $\sigma_y^2 = \epsilon_y\beta_y$.


**Q1) What is minimum dispersion in this lattice? At what location does this occur: focusing quad, defocusing quad, or drift?  
If we compare this lattice to an identical FODO cell without the dipole, would the horizontal beam size at this location be larger or smaller? What about the vertical beam size?**


In [ ]:
fodo_dispersion_summary = pd.DataFrame({
    "quantity": [
        "min eta_x [m]", "location of min eta_x", "max eta_x [m]", "location of max eta_x",
        "Qx", "Qy", "chromaticity Cx=dQx/ddelta", "chromaticity Cy=dQy/ddelta",
    ],
    "value": [
        np.min(tw_fodo_bend.dx),
        tw_fodo_bend.name[np.argmin(tw_fodo_bend.dx)],
        np.max(tw_fodo_bend.dx),
        tw_fodo_bend.name[np.argmax(tw_fodo_bend.dx)],
        tw_fodo_bend.qx,
        tw_fodo_bend.qy,
        tw_fodo_bend.dqx,
        tw_fodo_bend.dqy,
    ],
})
fodo_dispersion_summary


**Q2) Assuming a 0.1% momentum spread ($\delta = \frac{\Delta p}{p_0}=0.001$) in the beam, what is the horizontal beam size we expect in the focusing quadrupole QF? How does this compare to our beam size without energy spread?  
What is the vertical beam size in QF with and without energy spread?**

Use the dispersion value at QF and the emittance above.


In [ ]:
beam_qf_delta0 = add_beam_size(tw_fodo_bend, sigma_dp=0.0).query("name == 'qf'")
beam_qf_delta = add_beam_size(tw_fodo_bend, sigma_dp=SIGMA_DP_DEFAULT).query("name == 'qf'")

pd.DataFrame({
    "": [r"$\sigma_x$ at QF [mm]", r"$\sigma_y$ at QF [mm]"],
    r"$\delta=0$": [beam_qf_delta0["sigma_x_mm"].iloc[0], beam_qf_delta0["sigma_y_mm"].iloc[0]],
    r"$\delta=0.001$": [beam_qf_delta["sigma_x_mm"].iloc[0], beam_qf_delta["sigma_y_mm"].iloc[0]],
})


|           | $\delta = 0$ | $\delta = 0.001$ |
|---        |---           |---                |
| $\sigma_x$ |              |                  |
| $\sigma_y$ |              |                  |


### B) Designing a Zero-Dispersion Insert

The next model is a compact double-bend-achromat-like section with two 18-degree bends. It starts with zero incoming dispersion. First, all quadrupole strengths are set to zero so you can see the dispersion generated by the bends. Then Xsuite varies the central quadrupole strength to minimize the outgoing $\eta_x$ and $\eta_x'$.


In [ ]:
def make_dba_section(q1=0.0, q2=0.0, q3=0.0, bend_angle_deg=18.0):
    bend_length = 1.0
    angle = np.deg2rad(bend_angle_deg)
    h = angle / bend_length
    elements = [
        xt.Drift(length=0.5),
        xt.RBend(length=bend_length, angle=angle, k0=h, h=h),
        xt.Drift(length=0.5),
        xt.Quadrupole(length=0.3, k1=q2),
        xt.Drift(length=0.7),
        xt.Quadrupole(length=0.3, k1=q1),
        xt.Drift(length=0.7),
        xt.Quadrupole(length=0.3, k1=q3),
        xt.Drift(length=0.5),
        xt.RBend(length=bend_length, angle=angle, k0=h, h=h),
        xt.Drift(length=0.5),
    ]
    names = ["d0", "b1", "d1", "q2", "d2", "q1", "d3", "q3", "d4", "b2", "d5"]
    return build_line(elements, names)


def twiss_from_zero_dispersion(line):
    return line.twiss(
        method="4d",
        betx=10.0, alfx=0.0,
        bety=10.0, alfy=0.0,
        dx=0.0, dpx=0.0,
        dy=0.0, dpy=0.0,
    )

dba_zero_quads = make_dba_section(q1=0.0, q2=0.0, q3=0.0)
tw_dba_zero_quads = twiss_from_zero_dispersion(dba_zero_quads)

plot_optics(tw_dba_zero_quads, "Double-bend section with all quadrupoles off");
twiss_table(tw_dba_zero_quads)


**Q3) What are $\eta_x$ and $\eta_x'$ at the end of the cell?**


In [ ]:
pd.DataFrame({
    "end quantity": [r"$\eta_x$ [m]", r"$\eta_x'$"],
    "value": [tw_dba_zero_quads.dx[-1], tw_dba_zero_quads.dpx[-1]],
})


Now turn on the middle quadrupole (`q1`). The cell below solves for the central quadrupole strength that makes the final dispersion as close to zero as possible.


In [ ]:
from scipy.optimize import minimize_scalar


def achromat_penalty(q1):
    tw = twiss_from_zero_dispersion(make_dba_section(q1=q1, q2=0.0, q3=0.0))
    return tw.dx[-1] ** 2 + tw.dpx[-1] ** 2

opt_q1 = minimize_scalar(achromat_penalty, bounds=(-5.0, 5.0), method="bounded")
q1_achromat = opt_q1.x

dba_achromat = make_dba_section(q1=q1_achromat, q2=0.0, q3=0.0)
tw_dba_achromat = twiss_from_zero_dispersion(dba_achromat)

plot_optics(tw_dba_achromat, "Central quadrupole adjusted to cancel outgoing dispersion");
pd.DataFrame({
    "quantity": ["optimized q1 k1 [1/m^2]", "final eta_x [m]", "final eta_x'"],
    "value": [q1_achromat, tw_dba_achromat.dx[-1], tw_dba_achromat.dpx[-1]],
})


**Q4) What is the strength you found?** Give your answer with at least two decimal places.

$k_1 =$


The flanking quadrupoles below add transverse focusing while preserving the dispersion-cancellation idea. This makes a simple DBA-like transport section useful for studying where dispersion becomes largest and where momentum aperture is tightest.


In [ ]:
dba_focused = make_dba_section(q1=q1_achromat, q2=1.33, q3=-1.59)
tw_dba_focused = twiss_from_zero_dispersion(dba_focused)
beam_dba = add_beam_size(tw_dba_focused, sigma_dp=0.0)

plot_optics(tw_dba_focused, "Focused double-bend achromat-like section");
pd.DataFrame({
    "quantity": ["max eta_x [m]", "location", "max beta_x [m]", "max sigma_x for delta=0 [mm]"],
    "value": [
        np.max(tw_dba_focused.dx),
        tw_dba_focused.name[np.argmax(tw_dba_focused.dx)],
        np.max(tw_dba_focused.betx),
        np.max(beam_dba["sigma_x_mm"]),
    ],
})


**Q5) What is the maximum dispersion in this cell?**

$\eta_x =$


#### Momentum Acceptance Limited by Dispersion

Because the effect of dispersion depends on the beam momentum spread, it presents a limit to the amount of spread that can be transported without losing particles.

**Q6) Assuming the vacuum chamber is a pipe with 2.5 cm radius, what is the largest momentum spread that can be tolerated before particles start to hit the chamber wall?**

Use $\sigma_x^2 = \epsilon_x \beta_x+\eta_x^2\left(\frac{\Delta p}{p_0}\right)^2$.

**Q7) If momentum spread exceeds this limit, where in the cell will beam loss occur?**


In [ ]:
pipe_radius = 0.025  # m
aperture_df = tw_dba_focused.to_pandas()[["name", "s", "betx", "dx"]].copy()
aperture_df["sigma_x_delta0_m"] = np.sqrt(GEOMETRIC_EMITTANCE * aperture_df["betx"])
aperture_df["max_sigma_dp_before_25mm"] = np.sqrt(
    np.maximum(pipe_radius ** 2 - aperture_df["sigma_x_delta0_m"] ** 2, 0.0)
) / np.abs(aperture_df["dx"].replace(0, np.nan))
aperture_df.sort_values("max_sigma_dp_before_25mm").head(5)


### C) Chromaticity in a Ring

To study tune and chromaticity, the code below builds a closed ring from 20 identical bending/focusing cells. The total bend is $2\pi$, so this is a simple storage-ring model. Although parts of an achromat can be dispersion-free, a ring still has chromaticity: off-momentum particles feel different effective focusing and therefore have different tunes.


In [ ]:
def make_chromatic_ring(n_bends=20, kq=0.6):
    elements = []
    names = []
    bend_angle = 2 * np.pi / n_bends
    bend_length = 1.0
    h = bend_angle / bend_length
    for i in range(n_bends):
        tag = f"c{i + 1}"
        elements += [
            xt.Drift(length=0.5),
            xt.Quadrupole(length=0.3, k1=kq),
            xt.Drift(length=0.5),
            xt.Bend(length=bend_length, angle=bend_angle, k0=h, h=h),
            xt.Drift(length=0.5),
            xt.Quadrupole(length=0.3, k1=-kq),
            xt.Drift(length=0.5),
        ]
        names += [f"d1_{tag}", f"qf_{tag}", f"d2_{tag}", f"b_{tag}", f"d3_{tag}", f"qd_{tag}", f"d4_{tag}"]
    return build_line(elements, names)

chromatic_ring = make_chromatic_ring(n_bends=20, kq=0.6)
tw_ring = chromatic_ring.twiss(method="4d", compute_chromatic_properties=True)

plot_optics(tw_ring, "Periodic optics and dispersion in the chromaticity ring");
pd.DataFrame({
    "quantity": ["Qx", "Qy", "chromaticity Cx=dQx/ddelta", "chromaticity Cy=dQy/ddelta", "max eta_x [m]"],
    "value": [tw_ring.qx, tw_ring.qy, tw_ring.dqx, tw_ring.dqy, np.max(tw_ring.dx)],
})


**Q8) Record x and y tunes to 3 significant figures:**

$\nu_x =$  
$\nu_y =$


**Q9) In our FODO cell, we apply equal focusing strengths in both planes and as a result, $\nu_x=\nu_y$. Looking at the beta functions for one bending cell, explain why you would not necessarily expect the tunes to be equal.**


**Q10) For a 0.1% momentum spread in the beam, what is the spread of tunes due to chromaticity?**

Use $\Delta \nu = C\frac{\Delta p}{p_0}$.

$C_x =$  
$C_y =$  
$\Delta \nu_x =$  
$\Delta \nu_y =$


In [ ]:
sigma_dp = SIGMA_DP_DEFAULT
pd.DataFrame({
    "quantity": ["Cx", "Cy", "Delta Qx for sigma_dp=0.001", "Delta Qy for sigma_dp=0.001"],
    "value": [tw_ring.dqx, tw_ring.dqy, tw_ring.dqx * sigma_dp, tw_ring.dqy * sigma_dp],
})


#### Momentum Acceptance Limit Due to Chromaticity

When operating a ring, we choose the tune to avoid resonant conditions. A resonance line satisfies $m\nu_x+n\nu_y=p$ for integers $m$, $n$, and $p$. The order of the resonance is $|m|+|n|$. Here we assume that the ring must avoid all resonances of order 3 and below.


In [ ]:
def resonance_segments(max_order=3, integer=(0, 0), lines=(1, 1, 1, 1)):
    pval = 40
    p_values = np.arange(0, pval + 1)
    qxmin, qymin = integer[0], integer[1]
    qxmax, qymax = qxmin + 1, qymin + 1
    segments = []

    for order in range(1, max_order + 1):
        m_values = np.linspace(-order, order, 2 * order + 1)
        n1_values = order - np.abs(m_values)
        n2_values = -n1_values
        for m, n1, n2 in zip(m_values, n1_values, n2_values):
            for p in p_values:
                if n1 == 0 and lines[1] and m != 0:
                    x = p / m
                    if qxmin - 0.01 <= x <= qxmax + 0.01:
                        segments.append((order, [x, x], [qymin, qymax], f"{int(m)}νx = {int(p)}"))
                elif m == 0 and lines[0] and n1 != 0:
                    for n in [n1, n2]:
                        y = p / n
                        if qymin - 0.01 <= y <= qymax + 0.01:
                            segments.append((order, [qxmin, qxmax], [y, y], f"{int(n)}νy = {int(p)}"))
                elif n1 != 0 and m != 0:
                    if lines[2]:
                        denom = n2 if np.sign(m) > 0 else n1
                        y0 = p / denom - m * qxmin / denom
                        y1 = p / denom - m * qxmax / denom
                        if max(y0, y1) >= qymin - 0.01 and min(y0, y1) <= qymax + 0.01:
                            segments.append((order, [qxmin, qxmax], [y0, y1], f"{int(m)}νx + {int(denom)}νy = {int(p)}"))
                    if lines[3]:
                        denom = n1 if np.sign(m) > 0 else n2
                        y0 = p / denom - m * qxmin / denom
                        y1 = p / denom - m * qxmax / denom
                        if max(y0, y1) >= qymin - 0.01 and min(y0, y1) <= qymax + 0.01:
                            segments.append((order, [qxmin, qxmax], [y0, y1], f"{int(m)}νx + {int(denom)}νy = {int(p)}"))
    return segments


def plot_tune_footprint(nux, nuy, cx, cy, sigma_dp=0.001, resonance_order=3):
    delta_nux = cx * sigma_dp
    delta_nuy = cy * sigma_dp
    integer = (int(nux), int(nuy))
    colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}
    fig = go.Figure()

    for order, xs, ys, label in resonance_segments(max_order=resonance_order, integer=integer):
        fig.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            name=f"order {order}",
            legendgroup=f"order {order}",
            showlegend=not any(tr.name == f"order {order}" for tr in fig.data),
            line=dict(color=colors.get(order, "black"), width=1),
            hovertemplate=f"{label}<br>order {order}<extra></extra>",
        ))

    fig.add_trace(go.Scatter(
        x=[nux], y=[nuy], mode="markers", name="ring tune",
        marker=dict(color="black", size=9),
        hovertemplate="ring tune<br>νx = %{x:.6g}<br>νy = %{y:.6g}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=[nux - delta_nux, nux + delta_nux],
        y=[nuy - delta_nuy, nuy + delta_nuy],
        mode="lines+markers",
        name="chromatic footprint",
        line=dict(color="black", width=3),
        hovertemplate="chromatic footprint<br>νx = %{x:.6g}<br>νy = %{y:.6g}<extra></extra>",
    ))

    fig.update_layout(
        title="Resonance diagram",
        xaxis_title="horizontal tune",
        yaxis_title="vertical tune",
        xaxis=dict(range=[integer[0] - 0.01, integer[0] + 1.01]),
        yaxis=dict(range=[integer[1] - 0.01, integer[1] + 1.01]),
        template="plotly_white",
        width=720,
        height=650,
    )
    fig.show()
    return delta_nux, delta_nuy

# Change sigma_dp to test different momentum spreads.
sigma_dp_for_plot = 0.001
resonance_order = 3
plot_tune_footprint(tw_ring.qx, tw_ring.qy, tw_ring.dqx, tw_ring.dqy, sigma_dp=sigma_dp_for_plot, resonance_order=resonance_order);


**Q11) Assuming we can tolerate resonances of order 4 and above, but cannot operate on resonances of order 3 and below, how big of a momentum spread can we tolerate in this ring?**

Use the plot above. Change `sigma_dp_for_plot` until the black chromatic tune footprint crosses a third-order-or-lower resonance line. Give your answer to the nearest 0.1%.


**Q12) Comparing your answers to Q6 and Q11, is the momentum acceptance of this ring limited by chromaticity or by dispersion?**
